# Lab 7.4 &mdash; Locating the Failure

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Order the attribution ladder &mdash; the order IS the design
- Handle the run with three things wrong: the ladder must name the first
- Count the spend wasted downstream of each failure
- Choose between the step that fails most often and the step that costs most
- Diagnose a real failed run off a real trace

> **How this lab works.** You write real LangChain code &mdash; the agent under test, the callback
> handler that traces it, the typed verdict you grade. Fill every `BLANK`, then run the
> **Self-check** cell under each section. Those check the *objects you built* and the *recorded
> runs* shipped in the notebook, so they are deterministic and do not depend on the model.
> Cells marked **Run it for real** put your code in front of the sandbox model; that is the part
> worth watching, and it is never scored &mdash; scoring a live run would contradict Lab 7.1.

> **Without a trace, every one of these is &lsquo;the agent hallucinated&rsquo;.**
> Seven of the eight are not, and each has a different owner.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap, random, statistics
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because an eval lab makes a lot of calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 7 labs -- the same payment exceptions, now the
# subject of measurement rather than of engineering.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Module 1, Lab 1.2
# Real LangChain tools -- @tool turns a function into a tool object with a name, a schema
# and a description the model reads. Nothing to fill in; they are here so this notebook
# stands on its own and so the agent you measure is a real agent.

from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1005'.

    Use when you need the status, amount, counterparty or reason code of a specific
    payment. Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


EVAL_TOOLS = [lookup_payment, policy_for]
print("tools:", ", ".join(t.name for t in EVAL_TOOLS))

In [ ]:
# ------------------------------------------------- carried forward from Lab 7.2
# Your span store and your callback tracer, finished. Nothing to fill in -- they are here
# so this notebook stands alone, and so the last cell can diagnose a REAL failed run.

from langchain_core.callbacks import BaseCallbackHandler

class SpanStore:
    """Spans, and the links between them."""

    def __init__(self):
        self.spans, self.now = {}, 0.0

    def open_span(self, span_id, parent_id, name, kind):
        self.spans[span_id] = {"id": span_id, "parent": parent_id, "name": name, "kind": kind,
                               "t0": self.now, "t1": None, "tokens": 0, "status": "ok"}

    def close_span(self, span_id, tokens=0, status="ok"):
        span = self.spans.get(span_id)
        if span is None:
            return
        span["t1"], span["tokens"], span["status"] = self.now, tokens, status

    def ordered(self):
        return sorted(self.spans.values(), key=lambda s: (s["t0"], s["id"]))


class SpanTracer(BaseCallbackHandler):
    """The store, driven by LangChain's callbacks."""

    def __init__(self):
        self.store, self.t0 = SpanStore(), time.perf_counter()

    def _start(self, run_id, parent_run_id, name, kind):
        self.store.now = round(time.perf_counter() - self.t0, 3)
        self.store.open_span(str(run_id), str(parent_run_id) if parent_run_id else None,
                             name, kind)

    def _end(self, run_id, status="ok"):
        self.store.now = round(time.perf_counter() - self.t0, 3)
        self.store.close_span(str(run_id), status=status)

    def on_chain_start(self, serialized, inputs, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id,
                    kw.get("name") or (serialized or {}).get("name") or "chain", "chain")
    def on_chain_end(self, outputs, *, run_id=None, **kw):        self._end(run_id)
    def on_chain_error(self, error, *, run_id=None, **kw):        self._end(run_id, "error")
    def on_chat_model_start(self, serialized, messages, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id, "llm", "llm")
    def on_llm_start(self, serialized, prompts, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id, "llm", "llm")
    def on_llm_end(self, response, *, run_id=None, **kw):         self._end(run_id)
    def on_llm_error(self, error, *, run_id=None, **kw):          self._end(run_id, "error")
    def on_tool_start(self, serialized, input_str, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id, (serialized or {}).get("name") or "tool", "tool")
    def on_tool_end(self, output, *, run_id=None, **kw):          self._end(run_id)
    def on_tool_error(self, error, *, run_id=None, **kw):         self._end(run_id, "error")

print("carried forward: SpanStore, SpanTracer")

In [ ]:
# ------------------------------------------------- eight failed runs, read off their traces
# Each row is what a span tree told us about one failed run. Without it, all eight are
# reported the same way: "the agent got it wrong".

FAILED_RUNS = [
    {"id": "F1", "evidence_has_answer": False, "tool_error_ignored": False,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": False,
     "tokens_before_failure": 500,  "tokens_total": 2400},
    {"id": "F2", "evidence_has_answer": True,  "tool_error_ignored": True,
     "routed_to": "ledger", "should_route_to": "ledger", "constraints_dropped": False,
     "tokens_before_failure": 1200, "tokens_total": 1900},
    {"id": "F3", "evidence_has_answer": True,  "tool_error_ignored": False,
     "routed_to": "writer", "should_route_to": "sanctions", "constraints_dropped": False,
     "tokens_before_failure": 120,  "tokens_total": 1730},
    {"id": "F4", "evidence_has_answer": True,  "tool_error_ignored": False,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": True,
     "tokens_before_failure": 800,  "tokens_total": 2100},
    {"id": "F5", "evidence_has_answer": True,  "tool_error_ignored": False,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": False,
     "tokens_before_failure": 2000, "tokens_total": 2050},
    # three things wrong at once -- an ordered ladder must name the one that came first
    {"id": "F6", "evidence_has_answer": False, "tool_error_ignored": False,
     "routed_to": "writer", "should_route_to": "policy", "constraints_dropped": True,
     "tokens_before_failure": 400,  "tokens_total": 2600},
    {"id": "F7", "evidence_has_answer": True,  "tool_error_ignored": True,
     "routed_to": "policy", "should_route_to": "policy", "constraints_dropped": False,
     "tokens_before_failure": 1300, "tokens_total": 2100},
    {"id": "F8", "evidence_has_answer": True,  "tool_error_ignored": True,
     "routed_to": "ledger", "should_route_to": "ledger", "constraints_dropped": False,
     "tokens_before_failure": 1250, "tokens_total": 1900},
]

def find(run_id: str) -> dict:
    return next(r for r in FAILED_RUNS if r["id"] == run_id)

print(f"{len(FAILED_RUNS)} failed runs to diagnose")

## Concept

&ldquo;The agent was wrong&rdquo; is not a diagnosis. It is what you are left with when you did not keep
the trace, and it lands on whoever owns the agent regardless of who owns the bug.

A run can have several things wrong with it. The one that came **first** is the cause; everything
after it was doomed anyway. So the ladder is not a set of checks &mdash; it is an *ordered* list, and
the order is the whole design.

## Section 1 &mdash; The ladder

Five rungs. Each test is given; what you decide is the order they run in, and what happens when
none of them fires.

In [ ]:
RUNG_TESTS = {
    "retrieval":     lambda r: not r["evidence_has_answer"],
    "tool contract": lambda r: r["tool_error_ignored"],
    "routing":       lambda r: r["routed_to"] != r["should_route_to"],
    "handoff":       lambda r: r["constraints_dropped"],
}

def ladder() -> list:
    """The four rungs, in the order they are checked.

    Order them the way the run happened: the earliest step in the pipeline first. Getting
    this backwards produces diagnoses that are true and useless -- you fix a real bug whose
    repair would not have changed the run, because the run was already doomed upstream.
    """
    # TODO: return the four keys of RUNG_TESTS as a list, in checking order.
    return BLANK


def attribute(run: dict) -> str:
    """Name the step that caused this failure."""
    for rung in ladder():
        if RUNG_TESTS[rung](run):
            return rung
    # TODO: every rung above checked out and the answer was still wrong. Name what is left.
    return BLANK


def owner(step: str) -> str:
    """Who picks this up, which is the reason the diagnosis matters at all."""
    return {"retrieval": "the corpus and the chunker",
            "tool contract": "whoever wrote the tool",
            "routing": "the supervisor's descriptions",
            "handoff": "the message between two agents",
            "generation": "the prompt, or the model"}[step]

In [ ]:
# --- Self-check: Section 1   (recorded runs -- no model call)
check("the ladder names every rung exactly once",
      lambda: sorted(ladder()) == sorted(RUNG_TESTS) and len(ladder()) == len(set(ladder())))
check("bad evidence is a retrieval failure",
      lambda: attribute(find("F1")) == "retrieval")
check("an ignored tool error is a tool contract failure",
      lambda: attribute(find("F2")) == "tool contract")
check("the wrong specialist is a routing failure",
      lambda: attribute(find("F3")) == "routing")
check("a dropped constraint is a handoff failure",
      lambda: attribute(find("F4")) == "handoff")
check("only when everything upstream was fine is it generation",
      lambda: attribute(find("F5")) == "generation",
      "one run out of eight -- and it is the diagnosis all eight would have received")
check("F6 HAS THREE THINGS WRONG AND IS ATTRIBUTED TO THE FIRST",
      lambda: attribute(find("F6")) == "retrieval",
      "fixing its routing would change nothing: the evidence was already wrong when it routed")
check("every diagnosis names an owner",
      lambda: all(owner(attribute(r)) for r in FAILED_RUNS))

def _diagnose():
    for r in FAILED_RUNS:
        step = attribute(r)
        print(f"  {r['id']}  {step:15} -> {owner(step)}")
guard(_diagnose)

## Section 2 &mdash; What the failure cost

Everything spent after the failing step answered the wrong question. That number is what turns a
diagnosis into a priority.

In [ ]:
def wasted(run: dict) -> int:
    """Tokens spent after the thing that had already gone wrong."""
    return run["tokens_total"] - run["tokens_before_failure"]


def waste_rate(run: dict) -> float:
    return wasted(run) / run["tokens_total"] if run["tokens_total"] else 0.0

In [ ]:
# --- Self-check: Section 2
check("an early failure wastes most of the run",
      lambda: waste_rate(find("F3")) > 0.9,
      "a misroute at 120 tokens leaves 1,610 spent on the wrong specialist")
check("a late failure wastes almost nothing",
      lambda: waste_rate(find("F5")) < 0.05,
      "the generation failure happened at the end, so nothing downstream was thrown away")
check("waste is never negative",
      lambda: all(wasted(r) >= 0 for r in FAILED_RUNS))
check("the earliest failures are the most expensive ones",
      lambda: waste_rate(find("F3")) > waste_rate(find("F4")) > waste_rate(find("F5")),
      "which is why the ladder is ordered upstream-first, and why routing is worth measuring")
check("more than half of everything these runs spent was spent after they had already failed",
      lambda: sum(wasted(r) for r in FAILED_RUNS)
              > 0.5 * sum(r["tokens_total"] for r in FAILED_RUNS))

## Section 3 &mdash; Which fix first

Group the failures, add up what each group costs, and let that choose the work. The step that
fails most often and the step that costs most are **different steps here**, so this is a decision
rather than a sort.

In [ ]:
def by_step() -> dict:
    """{step: {"runs": n, "wasted": tokens}} across every failed run."""
    out = {}
    for run in FAILED_RUNS:
        entry = out.setdefault(attribute(run), {"runs": 0, "wasted": 0})
        entry["runs"] += 1
        entry["wasted"] += wasted(run)
    return out


def fix_first() -> str:
    """Which step do you send someone to fix on Monday?"""
    # TODO: "runs" ranks by how often a step fails; "wasted" ranks by what its failures
    # cost. One of them puts three cheap late failures ahead of two expensive early ones.
    key = BLANK
    return max(by_step().items(), key=lambda kv: kv[1][key])[0]


def most_frequent() -> str:
    return max(by_step().items(), key=lambda kv: kv[1]["runs"])[0]

In [ ]:
# --- Self-check: Section 3
check("every failed run is accounted for exactly once",
      lambda: sum(v["runs"] for v in by_step().values()) == len(FAILED_RUNS))
check("the wasted tokens add up",
      lambda: sum(v["wasted"] for v in by_step().values())
              == sum(wasted(r) for r in FAILED_RUNS))
check("the most FREQUENT cause is the tool contract, three runs out of eight",
      lambda: most_frequent() == "tool contract" and by_step()["tool contract"]["runs"] == 3)
check("BUT THE MOST EXPENSIVE ONE IS RETRIEVAL, on two runs",
      lambda: fix_first() == "retrieval" and by_step()["retrieval"]["runs"] == 2,
      "two early failures throw away more than three late ones -- rank by cost, not by count")
check("the two answers really are different steps",
      lambda: fix_first() != most_frequent(),
      "which is the whole reason this is a decision and not a sort")
check("generation is the rarest cause, and the cheapest",
      lambda: by_step()["generation"]["runs"] == 1
              and by_step()["generation"]["wasted"] == min(v["wasted"]
                                                           for v in by_step().values()))
check("without the ladder every one of these is 'generation'",
      lambda: len(by_step()) > 1,
      "five different owners, one default diagnosis, and four teams who never hear about it")

def _priority():
    print(f"  {'step':16}{'runs':>6}{'wasted':>9}")
    print("  " + "-" * 32)
    for step, v in sorted(by_step().items(), key=lambda kv: -kv[1]["wasted"]):
        print(f"  {step:16}{v['runs']:>6}{v['wasted']:>9}")
    print(f"\n  fails most often : {most_frequent()}")
    print(f"  fix first        : {fix_first()} -- {owner(fix_first())}")
guard(_priority)

## Run it for real &mdash; diagnose a run you just broke

A tool that raises, an agent that carries on regardless, and your tracer watching. The ladder
reads `tool contract` off the trace rather than off a hand-written flag.

In [ ]:
if llm_ready():
    from langchain_core.tools import tool
    from langchain.agents import create_agent

    @tool
    def sanctions_check(counterparty: str) -> str:
        """Return the sanctions screening status for one counterparty name."""
        raise RuntimeError("screening service unavailable (503)")

    def _break_something():
        tracer = SpanTracer()
        agent = create_agent(
            model=get_llm(), tools=[lookup_payment, policy_for, sanctions_check],
            system_prompt="You are a payments analyst. Look up PMT-1005, screen its "
                          "counterparty, then say what to do.")
        result = None
        try:
            result = agent.invoke(
                {"messages": [("human", "What should we do about PMT-1005?")]},
                config={"callbacks": [tracer], "recursion_limit": 8})
        except Exception as exc:
            print(f"  the run raised: {type(exc).__name__}")

        # The failure shows up in one of two places, depending on whether the tool node
        # swallowed the exception and handed the model an error string instead.
        errored = [s for s in tracer.store.ordered() if s["status"] == "error"]
        tool_errors = [m for m in (result or {}).get("messages", [])
                       if getattr(m, "type", "") == "tool"
                       and "error" in str(m.content).lower()]
        print(f"  {len(tracer.store.spans)} spans; {len(errored)} errored spans, "
              f"{len(tool_errors)} tool messages carrying an error")
        for s in errored:
            print(f"    error span: {s['name']} ({s['kind']})")
        observed = {"evidence_has_answer": True,
                    "tool_error_ignored": bool(errored or tool_errors),
                    "routed_to": "ledger", "should_route_to": "ledger",
                    "constraints_dropped": False}
        print(f"\n  ladder says: {attribute(observed)} -> {owner(attribute(observed))}")
    guard(_break_something)

## Run it for real &mdash; ask the model instead

Give the model the same evidence for F6 and see whether it reaches for an ordered ladder or for
the default.

In [ ]:
if llm_ready():
    def _ask_diagnosis():
        run = find("F6")
        reply = ask(
            "An agent run produced a wrong answer. Here is what the trace shows:\n"
            f"- the retrieved evidence did not contain the answer: {not run['evidence_has_answer']}\n"
            f"- a tool returned an error the agent ignored: {run['tool_error_ignored']}\n"
            f"- routed to {run['routed_to']}, should have been {run['should_route_to']}\n"
            f"- the handoff dropped a constraint: {run['constraints_dropped']}\n\n"
            "Name the ONE step that should be fixed first, and why.",
            system="Be brief and name a single step.")
        print("  model :", reply.strip()[:220])
        print(f"  ladder: {attribute(run)} -- {owner(attribute(run))}")
    guard(_ask_diagnosis)

### Read it

F6 has three things wrong with it, and only one of them is worth fixing first. If the model picks
routing or the handoff, it has picked a real bug whose repair would have changed nothing about
this run &mdash; the evidence was already wrong before either of them happened.

That is the value of an ordered ladder over a judgement: it is not smarter, it is just consistent,
and consistency is what lets you aggregate across a thousand runs and act on the total.

In [ ]:
score()

## Your turn

1. The ladder assumes each rung is observable. Which of the five would your current system be
   able to answer from its logs today? That list is your instrumentation backlog.
2. `wasted` counts tokens. Count seconds instead, using Lab 7.2's `self_time`, and see whether
   `fix_first` changes its mind. It usually does.
3. Add a rung for a failure this ladder cannot express &mdash; a case where the run was correct and
   the *question* was wrong. Where in the order does it go, and who owns it?